In [1]:
import numpy as np
import pandas as pd
import pathlib
from tqdm.auto import tqdm

import hydra
from omegaconf import DictConfig, OmegaConf

import torch
from torch_geometric import seed_everything

import ray

In [2]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

experiment = 220413
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [3]:
ray.init(num_cpus=24)

2022-04-15 07:36:02,063	INFO services.py:1412 -- View the Ray dashboard at http://127.0.0.1:8266


{'node_ip_address': '10.32.105.6',
 'raylet_ip_address': '10.32.105.6',
 'redis_address': None,
 'object_store_address': '/tmp/ray/session_2022-04-15_07-35-56_534598_2064909/sockets/plasma_store',
 'raylet_socket_name': '/tmp/ray/session_2022-04-15_07-35-56_534598_2064909/sockets/raylet',
 'webui_url': '127.0.0.1:8266',
 'session_dir': '/tmp/ray/session_2022-04-15_07-35-56_534598_2064909',
 'metrics_export_port': 54054,
 'gcs_address': '10.32.105.6:60291',
 'address': '10.32.105.6:60291',
 'node_id': '30a8a7dd94f0896f6022c225a35e17859f516603ca984c01e77842ca'}

In [4]:
import wandb
api = wandb.Api()
entity, project = "cardiors", "recordgraphs"  # set to your entity and project 
runs = api.runs(entity + "/" + project) 

In [5]:
run_list = []
for run in tqdm(runs): 
    run_list.append(
        {
            "id": run.path[-1], 
            "name": run.name,
            "tags": run.tags,
            "config": {k: v for k,v in run.config.items() if not k.startswith('_')},
            "summary": run.summary._json_dict,
            "path": None if "best_checkpoint" not in run.config.keys() else str(pathlib.Path(run.config["best_checkpoint"]).parent.parent)
        }
    )

  0%|          | 0/2270 [00:00<?, ?it/s]

In [6]:
runs_df = pd.DataFrame(run_list)

In [7]:
tag = "220413"
runs_df = runs_df[runs_df.tags.astype(str).str.contains(tag)].query("path==path")

## Process Predictions

In [8]:
name_dict = {
    "CovariatesOnlyTraining_['age_at_recruitment_f21022_0_0', 'sex_f31_0_0']_None_MLPHead": "Identity(AgeSex)+MLP",
    "RecordsIdentityTraining_[]_None_MLPHead": "Identity(Records)+MLP",
    "RecordsGraphTraining_[]_HeteroGNN_MLPHead": "GNN(Records)+MLP",
    "RecordsIdentityTraining_['age_at_recruitment_f21022_0_0', 'sex_f31_0_0']_None_MLPHead": "Identity(AgeSex+Records)+MLP",
    "RecordsGraphTraining_['age_at_recruitment_f21022_0_0', 'sex_f31_0_0']_HeteroGNN_MLPHead": "GNN(AgeSex+Records)+MLP"
}

In [9]:
id_vars = ["eid", "model", "partition", "split"]

In [10]:
out_path = f"{experiment_path}/loghs"
pathlib.Path(out_path).mkdir(parents=True, exist_ok=True)

In [11]:
@ray.remote
def prepare_predictions(in_path, out_path):
    temp = pd.read_feather(in_path).rename(columns={"index": "eid"}).reset_index(drop=True)
    temp["model"] = (temp.module.astype(str) + "_" + temp.covariate_cols.astype(str) + "_" + temp.encoder.astype(str) + "_" + temp["head"].astype(str)).astype("category")
    temp = temp.replace({"model":name_dict}).drop(columns=["module", "encoder", "head", "covariate_cols", "record_cols"])
    for c in id_vars: temp[c] = temp[c].astype("category")
    
    model = temp.model.unique()[0]
    partition = temp.partition.unique()[0]
    for split in ["train", "valid", "test"]:
        fp_out = f"{out_path}/{model}/{partition}"
        pathlib.Path(fp_out).mkdir(parents=True, exist_ok=True)
        temp.query("split==@split").reset_index(drop=True).to_feather(f"{fp_out}/{split}.feather")

In [12]:
for p in tqdm(runs_df.path): 
    prepare_predictions.remote(f"{p}/predictions/predictions.feather", out_path)

  0%|          | 0/44 [00:00<?, ?it/s]

## Get Endpoints

In [15]:
sample_loghs = pd.read_feather(f"{runs_df.path.iloc[0]}/predictions/predictions.feather").rename(columns={"index": "eid"}).reset_index(drop=True)

In [16]:
endpoints = [c for c in sample_loghs.columns if "OMOP_" in c or "phecode_" in c]

In [17]:
textfile = open(f"{experiment_path}/endpoints.txt", "w")
for element in endpoints:
    textfile.write(element + "\n")
textfile.close()

In [18]:
phecode_defs = pd.read_feather(f"{output_path}/phecode_defs_220306.feather").assign(phecode_label = lambda x: x.endpoint + " - " + x.phecode_string)

In [20]:
ray.shutdown()

In [26]:
runs_df.path.iloc[0]

'/sc-projects/sc-proj-ukb-cvd/results/models/RecordGraphs/3oq3n7by'

In [27]:
test = f"{experiment_path}/predictions"

In [29]:
!rm -rf $test

In [ ]:
predictions_list = []

for p in tqdm(runs_df.path):
    temp = read_prepare_feather(f"{p}/predictions/predictions.feather")
    predictions_list.append(temp)
    
predictions = pd.concat(predictions_list, axis=0).reset_index(drop=True)

In [11]:
for c in ["model", "split"]: predictions[c] = predictions[c].astype("category")

### Save wide format

In [ ]:
predictions.reset_index(drop=True).to_parquet(path=f"{pred_path}/predictions_wide", partition_cols=["model", "partition", "split"])

In [39]:
predictions.reset_index(drop=True).to_feather(f"{pred_path}/predictions_wide.feather")

In [1]:
predictions.info()

NameError: name 'predictions' is not defined

### Save long format

In [41]:
endpoint_cols = [c for c in predictions.columns if "OMOP" in c or "phecode" in c]

In [ ]:
predictions_long = predictions.melt(
    id_vars=id_vars, 
    value_vars=endpoint_cols, 
    var_name="endpoint", 
    value_name="logh")

In [ ]:
predictions_long.info()

In [ ]:
predictions_long["endpoint"] = predictions_long["endpoint"].astype("category")

In [ ]:
predictions_long["eid"] = predictions_long["eid"].astype(np.int32)

In [ ]:
# save long format
endpoints = predictions_long.endpoint.unique().tolist()
for e in tqdm(endpoints):
    predictions_long.query("endpoint==@e").reset_index(drop=True).to_feather(f"{pred_path}/predictions_{e}.feather")

In [52]:
print(sorted(endpoints))

['OMOP_4306655', 'phecode_001', 'phecode_002', 'phecode_002-1', 'phecode_003', 'phecode_004', 'phecode_004-1', 'phecode_005', 'phecode_005-1', 'phecode_006', 'phecode_006-2', 'phecode_007', 'phecode_007-1', 'phecode_009', 'phecode_010', 'phecode_015', 'phecode_016', 'phecode_016-1', 'phecode_019', 'phecode_023', 'phecode_027', 'phecode_030', 'phecode_033', 'phecode_050', 'phecode_050-5', 'phecode_052', 'phecode_052-1', 'phecode_052-3', 'phecode_052-31', 'phecode_052-32', 'phecode_052-4', 'phecode_054', 'phecode_056', 'phecode_059', 'phecode_059-1', 'phecode_061', 'phecode_070', 'phecode_074', 'phecode_084', 'phecode_084-1', 'phecode_084-2', 'phecode_086', 'phecode_088', 'phecode_089', 'phecode_089-1', 'phecode_089-2', 'phecode_089-3', 'phecode_091', 'phecode_092', 'phecode_092-2', 'phecode_095', 'phecode_098', 'phecode_100', 'phecode_100-1', 'phecode_101', 'phecode_101-1', 'phecode_101-2', 'phecode_101-21', 'phecode_101-4', 'phecode_101-41', 'phecode_101-42', 'phecode_101-6', 'phecode_

In [ ]:
predictions_long.index.dtype

In [35]:
predictions_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2514812300 entries, 0 to 2514812299
Data columns (total 6 columns):
 #   Column     Dtype   
---  ------     -----   
 0   eid        int32   
 1   split      category
 2   partition  category
 3   model      category
 4   endpoint   category
 5   logh       float32 
dtypes: category(4), float32(1), int32(1)
memory usage: 30.4 GB


In [30]:
predictions_long

,eid,split,partition,model,endpoint,logh
0,1917840,test,11,GNN(AgeSex+Records)+MLP,OMOP_4306655,-3.753446
1,1917853,test,11,GNN(AgeSex+Records)+MLP,OMOP_4306655,-1.636114
2,1917862,test,11,GNN(AgeSex+Records)+MLP,OMOP_4306655,-3.551745
3,1917871,test,11,GNN(AgeSex+Records)+MLP,OMOP_4306655,-2.610802
4,1917886,test,11,GNN(AgeSex+Records)+MLP,OMOP_4306655,-5.152038
...,...,...,...,...,...,...
2226853624,4079717,test,1,GNN(Records)+MLP,phecode_164-3,-8.673800
2226853625,4079724,test,1,GNN(Records)+MLP,phecode_164-3,-8.425591
2226853626,4079738,test,1,GNN(Records)+MLP,phecode_164-3,-5.308914
2226853627,4079746,test,1,GNN(Records)+MLP,phecode_164-3,-8.673800


In [20]:
1+1

2